# BP3 Gate 3 — Model / Classifier Benchmark & Champion Selection
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Why this notebook exists
BP3 Gate 2's real run (2026-09-22) tagged every real CFPB row with `intervention_required` /
`exclusion_reason` and wrote the Gold layer
(`data/processed/cfpb_intervention_escalation_gold.parquet`) — 815,453 real rows trainable
(10,511 `intervention_required=1`, 804,942 `=0`), 233,122 excluded (never dropped from the Gold
layer, only flagged non-trainable). This notebook is the first to actually train and benchmark
classifiers on that target.

## Real class imbalance — this drives every metric choice below
`positive_class_ratio_of_trainable` = **1.29%** (10,511 / 815,453) — roughly **76.6:1**
negative:positive. Master Plan Section 7's own BP3 methodology rule is explicit here: *"report
ROC-AUC, PR-AUC, recall and calibration, never a bare accuracy figure on an imbalanced target."*
This notebook follows that literally — **accuracy is never computed or reported anywhere below**,
not even alongside the real metrics as a curiosity (a trivial always-predict-0 classifier already
scores 98.71% accuracy on this target, so accuracy would be actively misleading here, not merely
uninformative). Champion selection uses mean CV **PR-AUC (average precision)** — the metric this
project's own Master Plan names for exactly this situation, and more informative than ROC-AUC
under this level of imbalance since it does not credit a classifier for the very large,
easy-to-get-right negative class. ROC-AUC and recall are also computed and reported per the same
Master Plan sentence. Calibration is Gate 4's own named deliverable (Master Plan Section 8's Gate
table: Gate 3 = "benchmarked model set, champion selected"; Gate 4 = "bootstrap CI, calibration,
confusion matrix, SHAP sample") — deferred there, not silently dropped.

## Candidate models — 5, not 6 (user instruction)
Per explicit instruction, this benchmark uses **5 candidates, including one basic/baseline
model**, not the Master Plan's full 6-model set: `logistic_regression` (the basic/baseline model),
`random_forest`, `hist_gradient_boosting`, `xgboost`, `lightgbm`. `catboost` is not attempted at
all — BP2 Gate 3 already root-caused a real, unresolved upstream incompatibility on this exact
installed scikit-learn version (Lesson #22: `cross_validate()`'s post-`clone()` identity check can
never pass for a `CatBoostClassifier` built with a `cat_features` constructor argument) and the
user's own resolution there was to drop CatBoost from BP2's candidate set entirely rather than
carry a special-cased workaround — the same choice is made here from the start rather than
re-discovering the identical failure.

## Feature set — reused unmodified from Gate 2 (HYPER)
`src/features/bp3_escalation_features.py` (built at Gate 2) already defines
`FEATURE_COLS_CATEGORICAL` (`Product, Sub-product, Issue, Sub-issue, State, Submitted via` — no
`common_taxonomy_bucket`; BP3 does not integrate BANKING77), `COMPANY_COL`, and `BARRED_COLUMNS`
— imported here unmodified rather than re-declared, avoiding the exact
triplicate-then-retrofit pattern BP2 hit before its own Gate 6. The Gold layer's categorical
columns already have Gate 2's explicit null-sentinel fill applied
(`MISSING_SUB_PRODUCT`/`MISSING_SUB_ISSUE`/`MISSING_STATE`) — this notebook live-re-verifies that
live-read null counts are zero rather than assuming Gate 2's fill still holds.

## What this notebook does
1. Live-verifies the Gold layer's real row accounting against Gate 2's documented config block
   (drift check, never trusts a baked-in count).
2. Loads only the trainable rows (`intervention_required` not null) and the approved feature
   columns + target from the Gold parquet via Polars (not `pandas.read_parquet`/
   `polars.to_pandas()` — same avoidable-`pyarrow`-dependency caution BP1/BP2 Gate 3 both recorded).
3. Builds a single stratified train/test split (CFPB has no provided split — Gate 1 policy already
   recorded this: fresh random split, stratified by `intervention_required`, `random_state=42`).
4. Benchmarks the 5 candidates under **identical CV folds** (Gate 3's own exit criterion), scored
   on `average_precision` (PR-AUC, the champion-selection metric), `roc_auc`, `recall`, and
   `precision`/`f1` at the default 0.5 threshold (explicit threshold tuning is a Gate 5 decision-
   layer concern, not benchmarked here). `class_weight='balanced'` applied wherever each library's
   real installed API supports it for binary classification (LogisticRegression, RandomForest,
   LightGBM); XGBoost's sklearn API has no `class_weight` but does expose `scale_pos_weight`,
   built specifically for binary imbalance — computed live from the real train split's class ratio
   (never a guessed constant) and used here, a real improvement over BP1/BP2's binary-less
   multiclass candidates which had no such lever available.
   `hist_gradient_boosting` has no equivalent in this environment's installed sklearn version for
   either binary or multiclass — the same documented library-capability asymmetry BP1/BP2 Gate 3
   both recorded, not an oversight here either.
5. Selects the champion by mean CV PR-AUC.
6. Refits the champion once on the full train split, evaluates once on the real held-out test
   split (PR-AUC, ROC-AUC, recall/precision/F1 at 0.5, full 2x2 confusion matrix — never accuracy).

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; it does not run it.
- **Zero-fabrication**: every row count, drift check, null re-check, and metric below is computed
  live against the real Gold parquet — nothing is assumed from a prior gate's output without
  re-verifying.
- **WARP**: reads Parquet (not CSV) via Polars, sparse one-hot matrices, `float32` for the one
  dense conversion `hist_gradient_boosting` requires, `n_jobs`/CV settings loaded from
  `hardware_benchmark_summary.json`/`resource_limits.yaml` rather than hardcoded.
- **Proactive safety hardening (not reactive this time)**: BP3's real trainable row count
  (815,453) is within 0.2% of BP2 Gate 3's own trainable set (816,717) — the exact real dataset
  scale that caused a real Windows crash (Kernel-Power Event 41) on BP2 Gate 3's first delivered
  version, root-caused and comprehensively fixed as Lesson #21 in `LESSONS_LEARNED_APPLIED.md`.
  Rather than waiting to rediscover that risk on a second real machine, this notebook is built
  with Lesson #21's full hardening from the start: per-candidate CV concurrency caps
  (`hist_gradient_boosting` n_jobs=1, `random_forest` n_jobs=2, everyone else capped to
  `min(4, N_JOBS)`), a pre-candidate adaptive RAM-headroom check that forces `n_jobs=1` if live
  headroom has already fallen below 4GB, and a 20-second precautionary inter-candidate pause for
  sustained-CPU/thermal load (`psutil` has no portable Windows temperature reading, so this
  remains a documented precaution, not a measured response — unchanged from BP2 Gate 3's own
  honesty note on this point).
- **HYPER**: reuses `src/features/bp3_escalation_features.py`'s constants unmodified (see above)
  and `src/utils/bp1_config_sync.py` unmodified (fifth BP-gate combination to do so, counting
  BP3's own Gate 1/Gate 2).

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate3_cv_benchmark_results.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate3_champion_test_classification_report.json`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate3_champion_test_confusion_matrix.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/model_inventory_entry.json` (SR 11-7
  first-line record — Gate 3's own named compliance touchpoint)
- `configs/bp3_complaint_escalation_prediction.yaml` — new `gate3` block appended, `status` field
  gets `_gate3_confirmed` appended

## Prerequisites
BP3 Gates 1 and 2 must both have run for real (the Gold parquet and both config blocks are
prerequisites) — this notebook re-verifies their real recorded numbers rather than trusting them
blindly.

## If a structural check below fails
It raises `AssertionError` with the failing check named. If every candidate fails, there is
nothing to select a champion from and this notebook must not proceed by falling back to an
arbitrary default model.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Gate 3 model / classifier benchmark notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split  # noqa: E402
from sklearn.preprocessing import OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
)

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
BP3_CONFIG_PATH = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"

for p in (GOLD_PATH, BP3_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP3 Gates 1-2 both completed for real."
        )

# ============================================================
# SECTION 4: Load Gate 1/2 policy - never hardcode what a prior gate already recorded
# ============================================================
with open(BP3_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

TARGET_COL = bp3_config["target_definition"]["primary_target"]
RANDOM_STATE = bp3_config["random_state"]
documented_trainable_rows = bp3_config["n_trainable_total"]
documented_n_positive = bp3_config["n_intervention_required"]
documented_n_negative = bp3_config["n_no_intervention_required"]
print(
    f"[OK] Gate 1/2 policy loaded: target='{TARGET_COL}', random_state={RANDOM_STATE}, "
    f"documented trainable rows (Gate 2)={documented_trainable_rows:,} "
    f"({documented_n_positive:,} positive / {documented_n_negative:,} negative)."
)

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
N_JOBS = hw_summary["recommended_configuration"]["recommended_n_jobs"]
print(f"[OK] Hardware benchmark recommendation loaded: n_jobs={N_JOBS} (never hardcoded)")

cv_settings = RESOURCE_LIMITS["cv"]
print(
    f"[OK] CV settings loaded: n_splits={cv_settings['n_splits']}, random_state={cv_settings['random_state']}"
)

# ============================================================
# SECTION 5: Load the real Gold layer via Polars (not pandas.read_parquet/polars.to_pandas() -
# same avoidable-pyarrow-dependency caution BP1/BP2 Gate 3 both recorded).
# ============================================================
gold_lazy = pl.scan_parquet(GOLD_PATH)

# Live re-check: Gate 2's explicit null-sentinel fill should mean zero real nulls remain in every
# candidate feature column on the Gold layer - verified live here, not assumed still true.
null_recheck = gold_lazy.select(
    [pl.col(c).is_null().sum().alias(c) for c in FEATURE_COLS_CATEGORICAL + [COMPANY_COL]]
).collect()
remaining_nulls = {c: int(null_recheck[c][0]) for c in null_recheck.columns if int(null_recheck[c][0]) > 0}
if remaining_nulls:
    print(f"[DRIFT DETECTED] Gold layer still has real nulls in candidate feature columns: {remaining_nulls}")
else:
    print(
        "[OK] Zero real nulls remain in any candidate feature column on the Gold layer - Gate 2's "
        "null-sentinel fill re-verified live, not assumed."
    )

select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL]
df_pl = gold_lazy.select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
for barred in BARRED_COLUMNS:
    assert (
        barred not in df_pl.columns
    ), f"[CHECK FAILED] barred column '{barred}' present in the loaded feature frame."

row_count_drift = df_pl.height != documented_trainable_rows
if row_count_drift:
    print(
        f"[DRIFT DETECTED] Live trainable row count ({df_pl.height:,}) != Gate 2's documented "
        f"count ({documented_trainable_rows:,})."
    )
else:
    print(f"[OK] Live trainable row count ({df_pl.height:,}) matches Gate 2's documented count exactly.")

class_dist = df_pl.group_by(TARGET_COL).agg(pl.len().alias("n")).sort("n", descending=True)
print(f"\n=== REAL intervention_required CLASS DISTRIBUTION (trainable rows only, n={df_pl.height:,}) ===")
print(class_dist)
n_positive_live = int(df_pl.filter(pl.col(TARGET_COL) == 1).height)
n_negative_live = int(df_pl.filter(pl.col(TARGET_COL) == 0).height)
positive_ratio = n_positive_live / df_pl.height
print(
    f"[FINDING] Real positive_class_ratio_of_trainable: {positive_ratio:.4f} "
    f"({n_negative_live / n_positive_live:.1f}:1 negative:positive). PR-AUC (average precision) "
    "is the champion-selection metric here, not accuracy, which is never computed in this "
    "notebook (Master Plan BP3 methodology rule)."
)

feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)
print(
    f"\n[OK] Built feature frame: {X_full.shape[0]:,} rows x {X_full.shape[1]} columns "
    f"({FEATURE_COLS_CATEGORICAL + [COMPANY_COL]})."
)

# ============================================================
# SECTION 6: Single stratified train/test split (CFPB has no provided split - Gate 1 policy)
# ============================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
print(
    f"\n[OK] Stratified train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,} "
    f"(test_size=0.20, random_state={RANDOM_STATE})."
)

n_train_positive = int((y_train == 1).sum())
n_train_negative = int((y_train == 0).sum())
scale_pos_weight = n_train_negative / n_train_positive
print(
    f"[OK] Live train-split class ratio: {n_train_negative:,} negative / {n_train_positive:,} "
    f"positive -> scale_pos_weight={scale_pos_weight:.2f} (computed live, never a guessed constant)."
)

# ============================================================
# SECTION 7: Shared preprocessing for all 5 candidates - one-hot the 6 low-cardinality categorical
# columns (fit on TRAIN only), frequency-encode Company (fit on TRAIN only, unseen companies at
# test time get frequency 0 - never fabricated, never test-set-derived).
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = (
    X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)
test_company_freq = (
    X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")
print(
    f"\n[OK] Shared preprocessed feature matrix: train={X_train_shared.shape}, test={X_test_shared.shape} "
    f"(one-hot on {len(FEATURE_COLS_CATEGORICAL)} columns + 1 Company frequency column, sparse)."
)

# ============================================================
# SECTION 8: Candidate model set - 5 candidates (user instruction: top 5 including one basic
# model), not the Master Plan's full 6. class_weight='balanced' / scale_pos_weight applied
# wherever this environment's real installed library API supports it for binary classification.
# ============================================================
CANDIDATES = {
    # The basic/baseline model, per explicit instruction.
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    # No class_weight/scale_pos_weight equivalent in this environment's installed
    # HistGradientBoostingClassifier - documented library-capability asymmetry, not an oversight
    # (same finding BP1/BP2 Gate 3 both recorded).
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    # XGBoost's sklearn API has no class_weight, but DOES expose scale_pos_weight for exactly this
    # binary-imbalance case - computed live in Section 6, not guessed.
    "xgboost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        n_jobs=1,
        verbosity=0,
        scale_pos_weight=scale_pos_weight,
        random_state=cv_settings["random_state"],
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100,
        class_weight="balanced",
        n_jobs=1,
        verbose=-1,
        random_state=cv_settings["random_state"],
    ),
}

# Real environment quirk already discovered on this exact installed sklearn version (BP1/BP2 Gate
# 3): HistGradientBoostingClassifier rejects sparse input. float32 keeps the transient densified
# matrix within the WARP RAM ceiling.
NEEDS_DENSE = {"hist_gradient_boosting"}

# ============================================================
# SECTION 8b: Proactive concurrency/thermal hardening, applied from the start (not reactively
# after an incident this time) - see this notebook's markdown "Proactive safety hardening" note.
# BP3's real trainable row count (815,453) is within 0.2% of the exact dataset scale that caused
# a real Windows crash on BP2 Gate 3's first delivered version (Lesson #21,
# LESSONS_LEARNED_APPLIED.md) - that lesson's full fix is reused here unmodified rather than
# risking the same incident on a second real machine.
# ============================================================
CV_N_JOBS_OVERRIDE = {
    "hist_gradient_boosting": 1,
    "random_forest": 2,
}
DEFAULT_CV_N_JOBS = min(4, N_JOBS)
COOLDOWN_SECONDS_BETWEEN_CANDIDATES = 20
MIN_HEADROOM_GB_BEFORE_CANDIDATE = 4.0

# ============================================================
# SECTION 9: Identical CV folds across every candidate (Gate 3 exit criterion). PR-AUC (average
# precision) is the champion-selection metric; ROC-AUC and recall are also computed per the
# Master Plan's explicit BP3 rule. Accuracy is never computed anywhere in this notebook.
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
SCORING = ["average_precision", "roc_auc", "recall", "precision", "f1"]

cv_results_rows = []
failed_candidates = []
candidate_names = list(CANDIDATES.keys())
for candidate_idx, (name, model) in enumerate(CANDIDATES.items()):
    X_cv, y_cv = X_train_shared, y_train
    if name in NEEDS_DENSE:
        X_cv = np.asarray(X_cv.todense(), dtype=np.float32)

    planned_cv_n_jobs = CV_N_JOBS_OVERRIDE.get(name, DEFAULT_CV_N_JOBS)
    headroom_before_gb = memory_headroom_gb(RESOURCE_LIMITS["ceilings"]["max_ram_fraction"])
    if headroom_before_gb < MIN_HEADROOM_GB_BEFORE_CANDIDATE:
        cv_n_jobs = 1
        print(
            f"[WARP] {name}: live headroom {headroom_before_gb}GB is below the "
            f"{MIN_HEADROOM_GB_BEFORE_CANDIDATE}GB pre-candidate threshold - forcing n_jobs=1 "
            f"(planned was {planned_cv_n_jobs})."
        )
    else:
        cv_n_jobs = planned_cv_n_jobs

    print(
        f"\n[BENCH] starting {name} ({candidate_idx + 1}/{len(candidate_names)}, "
        f"{cv_settings['n_splits']}-fold CV, n_jobs={cv_n_jobs}, threading backend, "
        f"headroom={headroom_before_gb}GB)..."
    )
    t0 = time.perf_counter()
    try:
        with joblib.parallel_backend("threading", n_jobs=cv_n_jobs):
            scores = cross_validate(model, X_cv, y_cv, cv=skf, scoring=SCORING, n_jobs=cv_n_jobs)
        elapsed = time.perf_counter() - t0
        row = {
            "model": name,
            "status": "OK",
            "elapsed_seconds": round(elapsed, 2),
            "cv_n_jobs_used": cv_n_jobs,
            "ram_headroom_gb_before_candidate": headroom_before_gb,
            "mean_average_precision": round(float(np.mean(scores["test_average_precision"])), 4),
            "std_average_precision": round(float(np.std(scores["test_average_precision"])), 4),
            "mean_roc_auc": round(float(np.mean(scores["test_roc_auc"])), 4),
            "mean_recall": round(float(np.mean(scores["test_recall"])), 4),
            "mean_precision": round(float(np.mean(scores["test_precision"])), 4),
            "mean_f1": round(float(np.mean(scores["test_f1"])), 4),
        }
        print(
            f"[BENCH] {name} done in {elapsed:.1f}s: mean average_precision={row['mean_average_precision']} "
            f"(+/- {row['std_average_precision']}), mean roc_auc={row['mean_roc_auc']}, "
            f"mean recall={row['mean_recall']}"
        )
    except Exception as e:  # noqa: BLE001 - continue gracefully on an individual model failure
        elapsed = time.perf_counter() - t0
        row = {
            "model": name,
            "status": f"FAILED: {type(e).__name__}: {e}",
            "elapsed_seconds": round(elapsed, 2),
            "cv_n_jobs_used": cv_n_jobs,
            "ram_headroom_gb_before_candidate": headroom_before_gb,
            "mean_average_precision": None,
            "std_average_precision": None,
            "mean_roc_auc": None,
            "mean_recall": None,
            "mean_precision": None,
            "mean_f1": None,
        }
        failed_candidates.append(name)
        print(
            f"[FAILED] {name} after {elapsed:.1f}s: {type(e).__name__}: {e} - "
            "continuing with remaining models."
        )
    cv_results_rows.append(row)
    del X_cv
    assert_within_ram_ceiling(RESOURCE_LIMITS)

    if candidate_idx < len(candidate_names) - 1:
        print(
            f"[WARP] cooling down {COOLDOWN_SECONDS_BETWEEN_CANDIDATES}s before the next "
            "candidate (precautionary pacing, not a measured thermal reading)..."
        )
        time.sleep(COOLDOWN_SECONDS_BETWEEN_CANDIDATES)

cv_results_df = pd.DataFrame(cv_results_rows)
print("\n=== CV BENCHMARK RESULTS (identical folds across all candidates) ===")
print(cv_results_df.to_string(index=False))

results_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
cv_results_df.to_csv(results_csv_path, index=False)
print(f"\n[SAVED] {results_csv_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Champion selection - highest mean CV average_precision (PR-AUC)
# ============================================================
passing_results = cv_results_df[cv_results_df["status"] == "OK"]
assert (
    len(passing_results) > 0
), "[CHECK FAILED] Every candidate model failed - nothing to select a champion from."
champion_row = passing_results.loc[passing_results["mean_average_precision"].idxmax()]
champion_name = champion_row["model"]
print(
    f"\n[RESULT] Champion: {champion_name} "
    f"(mean CV average_precision/PR-AUC={champion_row['mean_average_precision']})"
)

# ============================================================
# SECTION 11: Refit champion on the FULL train split, evaluate ONCE on the held-out test split.
# Probability-based metrics (PR-AUC, ROC-AUC) plus 0.5-threshold precision/recall/F1 and the full
# 2x2 confusion matrix - never accuracy.
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
X_train_final, X_test_final = X_train_shared, X_test_shared
if champion_name in NEEDS_DENSE:
    X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
    X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)

champion_model = CANDIDATES[champion_name]
print(
    f"\n[FINAL] Refitting champion ({champion_name}) on the full train split ({len(X_train_raw):,} rows)..."
)
t0 = time.perf_counter()
champion_model.fit(X_train_final, y_train)
fit_elapsed = time.perf_counter() - t0
print(
    f"[FINAL] Fit done in {fit_elapsed:.1f}s. Evaluating ONCE on the held-out test split "
    f"({len(X_test_raw):,} rows)..."
)

y_proba = champion_model.predict_proba(X_test_final)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

test_pr_auc = float(average_precision_score(y_test, y_proba))
test_roc_auc = float(roc_auc_score(y_test, y_proba))
test_report = classification_report(
    y_test, y_pred, output_dict=True, zero_division=0, target_names=["0", "1"]
)
test_recall = test_report["1"]["recall"]
test_precision = test_report["1"]["precision"]
test_f1 = test_report["1"]["f1-score"]
print(
    f"[FINAL] Held-out test (positive class = intervention_required==1): PR-AUC={test_pr_auc:.4f}, "
    f"ROC-AUC={test_roc_auc:.4f}, recall={test_recall:.4f}, precision={test_precision:.4f}, "
    f"f1={test_f1:.4f} (at the default 0.5 threshold - explicit threshold tuning is a Gate 5 "
    "decision-layer concern, not benchmarked here)."
)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["predicted_0", "predicted_1"])
print("\n[FINAL] Held-out test confusion matrix (2x2):")
print(cm_df)

# ============================================================
# SECTION 12: Write outputs (idempotent overwrite-in-place)
# ============================================================
report_json_path = ARTIFACTS_DIR / "gate3_champion_test_classification_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(test_report, f, indent=2)
print(f"\n[SAVED] {report_json_path.relative_to(PROJECT_ROOT)}")

cm_csv_path = ARTIFACTS_DIR / "gate3_champion_test_confusion_matrix.csv"
cm_df.to_csv(cm_csv_path)
print(f"[SAVED] {cm_csv_path.relative_to(PROJECT_ROOT)}")

model_inventory_entry = {
    "bp_id": "bp3",
    "gate": 3,
    "compliance_touchpoint": "Model inventory entry opened (SR 11-7 first-line record)",
    "model_name": champion_name,
    "model_family": "one-hot/frequency-encoded structured features + " + champion_name,
    "target_variable": TARGET_COL,
    "feature_columns": FEATURE_COLS_CATEGORICAL + [COMPANY_COL],
    "barred_columns": BARRED_COLUMNS,
    "training_data": (
        "Real CFPB extract, fresh stratified 80/20 split (CFPB provides no split of its own), "
        f"random_state={RANDOM_STATE}"
    ),
    "n_train_rows": len(X_train_raw),
    "n_test_rows": len(X_test_raw),
    "positive_class_ratio_of_trainable": round(positive_ratio, 4),
    "cv_folds": cv_settings["n_splits"],
    "cv_mean_average_precision": float(champion_row["mean_average_precision"]),
    "cv_mean_roc_auc": float(champion_row["mean_roc_auc"]),
    "held_out_test_pr_auc": test_pr_auc,
    "held_out_test_roc_auc": test_roc_auc,
    "held_out_test_recall": test_recall,
    "held_out_test_precision": test_precision,
    "held_out_test_f1": test_f1,
    "candidates_evaluated": list(CANDIDATES.keys()),
    "candidates_failed": failed_candidates,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "first-line record only - Gate 4 independent-style statistical validation not yet performed",
}
inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Write the Gate 3 config block (marker-based, order-independent, reuses
# bp1_config_sync.py unmodified)
# ============================================================
import re  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP3_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = (
    current_status_line.split('"')[1] + "_gate3_confirmed"
    if "_gate3_confirmed" not in current_status_line
    else current_status_line.split('"')[1]
)
status_text = re.sub(
    r"^status:.*$", f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE
)
BP3_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate3_marker = "# --- Gate 3 (Model/Classifier Benchmark) results (appended, idempotent overwrite) ---"
gate3_block_lines = [
    "gate3_model_benchmark:",
    f'  champion_model: "{champion_name}"',
    f"  cv_mean_average_precision: {champion_row['mean_average_precision']}",
    f"  cv_mean_roc_auc: {champion_row['mean_roc_auc']}",
    f"  held_out_test_pr_auc: {round(test_pr_auc, 4)}",
    f"  held_out_test_roc_auc: {round(test_roc_auc, 4)}",
    f"  held_out_test_recall: {round(test_recall, 4)}",
    f"  held_out_test_precision: {round(test_precision, 4)}",
    f"  held_out_test_f1: {round(test_f1, 4)}",
    f"  positive_class_ratio_of_trainable: {round(positive_ratio, 4)}",
    f"  candidates_evaluated: {list(CANDIDATES.keys())}",
    f"  candidates_failed: {failed_candidates}",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP3_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] {BP3_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate3_model_benchmark block)")

# ============================================================
# SECTION 14: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "at_least_one_candidate_passed": len(passing_results) > 0,
    "identical_cv_object_used_for_all_candidates": True,  # by construction - same `skf` instance, Section 9
    "champion_selected_by_mean_cv_average_precision": champion_name in CANDIDATES,
    "held_out_test_touched_exactly_once": True,  # by construction - Section 11 is the only test-set use
    "no_barred_column_in_feature_frame": all(b not in df_pl.columns for b in BARRED_COLUMNS),
    "no_accuracy_metric_computed_anywhere": "accuracy" not in SCORING,
    "cv_results_csv_written": results_csv_path.exists(),
    "classification_report_json_written": report_json_path.exists(),
    "confusion_matrix_csv_written": cm_csv_path.exists(),
    "model_inventory_entry_written": inventory_path.exists(),
    "bp3_config_yaml_updated": BP3_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 Gate 3 complete. Champion: {champion_name} "
    f"(held-out test PR-AUC={round(test_pr_auc, 4)}, ROC-AUC={round(test_roc_auc, 4)}, "
    f"recall={round(test_recall, 4)}). {len(failed_candidates)} candidate(s) failed: "
    f"{failed_candidates or 'none'}. Proceed to BP3 Gate 4 (Statistical Validation & "
    "Explainability) next."
)
